# Homework 3 - Human Activity Recognition

**Name:** Nakul Kumbria

**GitHub Username:** nakicool007

**USC ID:** 6748597520

## Setup

In [11]:
from pathlib import Path
import shutil

# Support Jupyter launched from the repository root or notebook directory.
cwd = Path.cwd().resolve()
root = next((candidate for candidate in (cwd, *cwd.parents)
             if (candidate / "data" / "AReM" / "bending1").is_dir()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the HW3 repository, with data/AReM present.")
data_root = root / "data" / "AReM"
processed_root = root / "data" / "processed"
processed_root.mkdir(parents=True, exist_ok=True)
activities = ("bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking")


## (a) AReM Dataset

I used the [AReM dataset from UCI](https://archive.ics.uci.edu/dataset/366/activity+recognition+system+based+on+multisensor+data+fusion). The files are in `data/AReM/`. There are 88 instances in seven folders: 7 in bending1, 6 in bending2, and 15 each in cycling, lying, sitting, standing, and walking.

Each file has a time column and six sensor columns: `avg_rss12`, `var_rss12`, `avg_rss13`, `var_rss13`, `avg_rss23`, and `var_rss23`.

The assignment says each instance has 480 observations. I checked the files and found that `sitting/dataset8.csv` has 479, while the other 87 files have 480. I kept the 479 values for that file instead of adding a made-up value.

Dataset citation: Palumbo, F., Gallicchio, C., Pucci, R., & Micheli, A. (2016). *Activity Recognition system based on Multisensor data fusion (AReM)*. UCI Machine Learning Repository. https://doi.org/10.24432/C5SS33. License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).


In [12]:
import csv
import math
from statistics import mean, median, stdev, quantiles
from IPython.display import HTML, display

series_names = [
    "avg_rss12", "var_rss12", "avg_rss13",
    "var_rss13", "avg_rss23", "var_rss23",
]

def read_time_series(path):
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        # Remove trailing commas and allow either commas or spaces.
        values = line.rstrip(",").split(",") if "," in line else line.split()
        if len(values) != 7:
            raise ValueError(f"Unexpected number of columns in {path}")
        rows.append([float(value) for value in values])

    if len(rows) != 480 and not (path.parent.name == "sitting" and path.name == "dataset8.csv" and len(rows) == 479):
        raise ValueError(f"Unexpected number of observations in {path}: {len(rows)}")
    if not all(math.isfinite(value) for row in rows for value in row):
        raise ValueError(f"Missing or non-finite values in {path}")
    # The first column is time, so only keep the six sensor columns.
    return list(zip(*(row[1:] for row in rows)))

In [13]:
source_files = [path for activity in activities
                for path in (data_root / activity).glob("dataset*.csv")]
assert len(source_files) == 88
row_counts = {}
for source in source_files:
    series = read_time_series(source)
    assert len(series) == 6
    row_counts[source.relative_to(data_root).as_posix()] = len(series[0])
assert sum(count == 480 for count in row_counts.values()) == 87
assert row_counts["sitting/dataset8.csv"] == 479
print("Validated 88 instances in seven activity folders, each with six sensor series.")
print("87 instances have 480 observations; sitting/dataset8.csv has 479 original observations.")


Validated 88 instances in seven activity folders, each with six sensor series.
87 instances have 480 observations; sitting/dataset8.csv has 479 original observations.


## (b) Training and Test Split

I used datasets 1 and 2 from bending1 and bending2 as test data. For the other five activities, I used datasets 1, 2, and 3 as test data. The remaining files are training data, as specified in the question.


In [14]:
train_files, test_files = [], []

for activity in activities:
    files = sorted(
        (data_root / activity).glob("dataset*.csv"),
        key=lambda path: int(path.stem.removeprefix("dataset")),
    )
    expected_count = 7 if activity == "bending1" else 6 if activity == "bending2" else 15
    actual_ids = {int(path.stem.removeprefix("dataset")) for path in files}
    assert actual_ids == set(range(1, expected_count + 1)), f"Unexpected files in {activity}"

    test_ids = {1, 2} if activity in {"bending1", "bending2"} else {1, 2, 3}
    for source in files:
        instance_id = int(source.stem.removeprefix("dataset"))
        if instance_id in test_ids:
            test_files.append(source)
        else:
            train_files.append(source)

assert len(train_files) == 69
assert len(test_files) == 19
assert set(train_files).isdisjoint(test_files)

### Saving the Split

I copied the files into `data/train/` and `data/test/`, keeping a folder for each activity. There are 69 training instances and 19 test instances. The code also checks that the copied files match the originals.


In [15]:
for split, files in (("train", train_files), ("test", test_files)):
    expected = {Path(source.parent.name) / source.name for source in files}
    split_root = root / "data" / split
    existing = {path.relative_to(split_root) for path in split_root.rglob("*.csv")}
    if existing - expected:
        raise ValueError(f"Unexpected files in {split_root}: {existing - expected}")
    for source in files:
        destination = split_root / source.parent.name / source.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        assert destination.read_bytes() == source.read_bytes()
    actual = {path.relative_to(split_root) for path in split_root.rglob("*.csv")}
    assert actual == expected, f"Incorrect membership in {split}"

print(f"Training instances: {len(train_files)}")
print(f"Test instances: {len(test_files)}")


Training instances: 69
Test instances: 19


## (c) Feature Extraction

### (i) Common Time-Domain Features

Time-domain features summarize a time series so we do not have to use every observation as a separate input. Some common features are:

- Minimum and maximum: the smallest and largest values.
- Mean and median: the average and middle value.
- Standard deviation and variance: how much the values vary.
- First and third quartiles: the 25th and 75th percentiles.
- Range and interquartile range: the differences between the extremes and between the quartiles.
- RMS and energy: measures of the size of the readings, using squared values.
- Mean absolute change: the average size of the change between consecutive readings.
- Autocorrelation and trend: how readings relate at different time lags and whether they increase or decrease over time.
- Peak and crossing counts: the number of local peaks or crossings of a reference value.
- Skewness and kurtosis: the asymmetry and tail heaviness of the distribution.

For this question, I used minimum, maximum, mean, median, standard deviation, and the first and third quartiles. Seven features for each of the six series gives 42 features per instance.


### (ii) Extracting the Features

I calculated the seven features for each sensor series without normalizing the values. I used sample standard deviation, which divides by n - 1, and linearly interpolated quartiles.

The numbers 1 through 6 in the column names follow the sensor order listed in part (a). For example, `min_1` is the minimum of `avg_rss12`, and `q1_6` is the first quartile of `var_rss23`.

The reader defined earlier handles files with spaces instead of commas and extra commas at the ends of rows. For `sitting/dataset8.csv`, I used all 479 available observations.


In [16]:
feature_rows = []
instance_info = []

for activity in activities:
    files = sorted(
        (data_root / activity).glob("dataset*.csv"),
        key=lambda path: int(path.stem.removeprefix("dataset")),
    )
    for source in files:
        instance = len(feature_rows) + 1
        features = {"Instance": instance}
        for number, values in enumerate(read_time_series(source), start=1):
            q1, _, q3 = quantiles(values, n=4, method="inclusive")
            statistics = {
                "min": min(values),
                "max": max(values),
                "mean": mean(values),
                "median": median(values),
                "std": stdev(values),
                "q1": q1,
                "q3": q3,
            }
            for statistic, value in statistics.items():
                features[f"{statistic}_{number}"] = value
        feature_rows.append(features)
        instance_info.append({
            "Instance": instance,
            "activity": activity,
            "file": source.name,
            "split": "test" if source in test_files else "train",
        })

feature_columns = list(feature_rows[0])[1:]
X_train = [[row[column] for column in feature_columns]
           for row, info in zip(feature_rows, instance_info) if info["split"] == "train"]
X_test = [[row[column] for column in feature_columns]
          for row, info in zip(feature_rows, instance_info) if info["split"] == "test"]
y_train = [info["activity"] for info in instance_info if info["split"] == "train"]
y_test = [info["activity"] for info in instance_info if info["split"] == "test"]

assert len(feature_rows) == 88 and len(feature_columns) == 42
assert len(X_train) == 69 and len(X_test) == 19
assert all(math.isfinite(row[column]) for row in feature_rows for column in feature_columns)
print(f"Feature dataset: {len(feature_rows)} instances, {len(feature_columns)} features")
print(f"Training set: {len(X_train)} instances; test set: {len(X_test)} instances")

Feature dataset: 88 instances, 42 features
Training set: 69 instances; test set: 19 instances


In [17]:
with (processed_root / "features.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["Instance"] + feature_columns)
    writer.writeheader()
    writer.writerows(feature_rows)

# Keep labels and split information separate from the 42 numeric features.
with (processed_root / "instance_metadata.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(instance_info[0]))
    writer.writeheader()
    writer.writerows(instance_info)

# Preview the first five instances. The full table is saved in features.csv.
preview_columns = ["Instance"] + feature_columns[:7] + ["..."] + feature_columns[-2:]
header = "".join(f"<th>{column}</th>" for column in preview_columns)
body = ""
for row in feature_rows[:5]:
    cells = []
    for column in preview_columns:
        value = "..." if column == "..." else str(row[column]) if column == "Instance" else f"{row[column]:.4f}"
        cells.append(f"<td>{value}</td>")
    body += "<tr>" + "".join(cells) + "</tr>"
display(HTML('<div style="overflow-x:auto"><table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

Instance,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,...,q1_6,q3_6
1,37.2500,45.0000,40.6248,40.5000,1.4770,39.2500,42.0000,...,0.0000,1.3000
2,38.0000,45.6700,42.8128,42.5000,1.4355,42.0000,43.6700,...,0.0000,1.3000
3,35.0000,47.4000,43.9545,44.3300,1.5588,43.0000,45.0000,...,0.0000,0.9400
4,33.0000,47.7500,42.1798,43.5000,3.6707,39.1500,45.0000,...,0.0000,1.0000
5,33.0000,45.7500,41.6781,41.7500,2.2435,41.3300,42.7500,...,0.0000,0.5000


The result has 88 rows and 42 feature columns, plus the instance number. I saved it in `data/processed/features.csv`. The activity labels, filenames, and train/test assignments are in `data/processed/instance_metadata.csv`.

The preview shows the first five instances and rounds the values to four decimal places. The CSV keeps the full precision.


### (iii) Standard Deviations and Bootstrap Confidence Intervals

I calculated the sample standard deviation of each feature across the 88 instances. For example, the standard deviation of `min_1` measures how much the minimum of the first sensor series changes between instances. This is different from the standard deviation within a single sensor series in part (ii).

For the 90% confidence intervals, I sampled 88 instances with replacement and recalculated the standard deviations. I repeated this 10,000 times and used the 5th and 95th percentiles as the interval limits. I used seed 42 so the results stay the same when I rerun the notebook. Each repetition uses the same sampled instance indices for all 42 features.


In [18]:
import random

n_bootstrap = 10_000
rng = random.Random(42)
n_instances = len(feature_rows)

feature_values = {
    column: [row[column] for row in feature_rows]
    for column in feature_columns
}
feature_std = {column: stdev(values) for column, values in feature_values.items()}
bootstrap_std = {column: [] for column in feature_columns}

# Centering preserves standard deviations and improves numerical stability.
centered_values = {
    column: [value - mean(values) for value in values]
    for column, values in feature_values.items()
}

for _ in range(n_bootstrap):
    sampled_indices = rng.choices(range(n_instances), k=n_instances)
    for column in feature_columns:
        sample = [centered_values[column][index] for index in sampled_indices]
        sample_sum = math.fsum(sample)
        sum_squares = math.fsum(value * value for value in sample)
        variance = (sum_squares - sample_sum ** 2 / n_instances) / (n_instances - 1)
        bootstrap_std[column].append(math.sqrt(max(variance, 0.0)))

print(f"Completed {n_bootstrap:,} bootstrap samples for {len(feature_columns)} features.")

Completed 10,000 bootstrap samples for 42 features.


In [19]:
def percentile(values, probability):
    ordered = sorted(values)
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    return ordered[lower] + (position - lower) * (ordered[upper] - ordered[lower])

std_ci_rows = []
for column in feature_columns:
    std_ci_rows.append({
        "feature": column,
        "std": feature_std[column],
        "ci_90_lower": percentile(bootstrap_std[column], 0.05),
        "ci_90_upper": percentile(bootstrap_std[column], 0.95),
    })

assert len(std_ci_rows) == 42
assert all(0 <= row["ci_90_lower"] <= row["ci_90_upper"] for row in std_ci_rows)
assert all(math.isfinite(value) for row in std_ci_rows for key, value in row.items() if key != "feature")

with (processed_root / "feature_std_ci.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(std_ci_rows[0]))
    writer.writeheader()
    writer.writerows(std_ci_rows)

header = "<tr><th>Feature</th><th>Standard deviation</th><th>90% CI lower</th><th>90% CI upper</th></tr>"
body = ""
for row in std_ci_rows:
    body += (
        f"<tr><td>{row['feature']}</td>"
        f"<td>{row['std']:.4f}</td>"
        f"<td>{row['ci_90_lower']:.4f}</td>"
        f"<td>{row['ci_90_upper']:.4f}</td></tr>"
    )
display(HTML("<table><thead>" + header + "</thead><tbody>" + body + "</tbody></table>"))

Feature,Standard deviation,90% CI lower,90% CI upper
min_1,9.5700,8.2646,10.7871
max_1,4.3944,3.3279,5.2927
mean_1,5.3357,4.7293,5.8694
median_1,5.4401,4.8076,5.9938
std_1,1.7722,1.5701,1.9484
q1_1,6.1536,5.5834,6.6379
q3_1,5.1389,4.3478,5.8331
min_2,0.0000,0.0000,0.0000
max_2,5.0627,4.6168,5.3996
mean_2,1.5742,1.3929,1.7045


The table gives the standard deviation and 90% bootstrap interval for each feature. Wider intervals mean more uncertainty in the estimate. I used all 88 instances for this part, including the training and test instances. The bootstrap assumes the instances are independent.

I saved these results in `data/processed/feature_std_ci.csv`.


### (iv) Selecting Three Important Features

I would choose mean, standard deviation, and maximum.

- Mean gives the average sensor reading and could help separate activities with different overall readings.
- Standard deviation shows how much the readings change. I would expect this to help distinguish movement, such as walking, from sitting or lying down.
- Maximum captures a high reading that might be missed by looking only at the average.

These features cover the average level, variation, and highest reading. Using them for all six series would give 18 features per instance. This is my initial choice; I would need to compare their performance on the training data to know whether they work best.


## Question 2 - Linear and Cubic Regression

The linear model uses $X$, while the cubic model uses $X$, $X^2$, and $X^3$. Both models are fitted to the same 100 observations using least squares.

### (a) Training RSS When the True Relationship Is Linear

The cubic regression will have a training RSS that is lower than or equal to the linear regression's. It can give the same fit as the linear model by setting the coefficients of $X^2$ and $X^3$ to zero. The extra terms can also fit some of the noise in the training data and reduce the RSS. So, even if the true relationship is linear, the cubic model's training RSS cannot be higher.

### (b) Test RSS When the True Relationship Is Linear

I would expect the linear regression to have a lower test RSS because the true relationship is already linear. The cubic model has extra terms that can fit noise in the training data, which may make its predictions worse on new data. This is what I would expect overall, but it is not guaranteed for every test sample.

### (c) Training RSS When the True Relationship Is Not Linear

The cubic regression will still have a training RSS that is lower than or equal to the linear regression's. Like in part (a), the cubic model includes the linear model, so adding the extra terms cannot make the least-squares fit worse on the training data. This holds whether the true relationship is linear or nonlinear.

### (d) Test RSS When the True Relationship Is Not Linear

There is not enough information to tell which model will have a lower test RSS. If the relationship has a lot of curvature and the cubic model fits that shape well, it could perform better. If the relationship is close to linear, the extra terms could mainly fit noise, and the linear model could perform better. We would need to know more about the relationship and noise level to decide.